# 0c. Fixed Points, Stop-Gradients, and Self-Distillation

The drifting loss is three lines of code:

```python
V      = compute_V(x, y_pos, y_neg)      # the drift, no gradient
target = (x + V).detach()                # frozen
loss   = mse(x, target)
```

Every subtlety in the method lives in that `.detach()`. This notebook is about why it is there, what it does to the
gradient, and why the resulting object is a **fixed-point iteration** rather than the minimisation of a loss surface.
If you have met SimSiam, BYOL, or consistency models, this will feel familiar — it is the same family of
bootstrapped-target objectives.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110

## 1. Fixed points

A **fixed point** of a map $T$ is a point with $T(\mathbf{x}^\star)=\mathbf{x}^\star$. If $T$ is a contraction, the
iteration $\mathbf{x}_{i+1}=T(\mathbf{x}_i)$ converges to it.

Drifting starts from the equilibrium condition $\mathbf{V}_{p,q_{\hat\theta}}=\mathbf{0}$ and rewrites it as a fixed
point in *function space*:

$$f_{\hat\theta}(\boldsymbol{\epsilon}) \;=\; f_{\hat\theta}(\boldsymbol{\epsilon}) + \mathbf{V}_{p,\,q_{\hat\theta}}\!\big(f_{\hat\theta}(\boldsymbol{\epsilon})\big)$$

which is trivially true iff $\mathbf{V}=\mathbf{0}$. The corresponding iteration is

$$f_{\theta_{i+1}}(\boldsymbol{\epsilon}) \;\leftarrow\; f_{\theta_i}(\boldsymbol{\epsilon}) + \mathbf{V}_{p,\,q_{\theta_i}}\!\big(f_{\theta_i}(\boldsymbol{\epsilon})\big)$$

and *that* is what the loss implements: it is a regression that tries to make the left side equal the right side.

In [ ]:
# a scalar fixed-point iteration, for intuition:  x = T(x) with T(x) = x + V(x)
def V_scalar(x, x_star=2.0, k=0.6):
    return k * (x_star - x)          # vanishes exactly at x_star

for x0 in [-3.0, 0.0, 6.0]:
    xs = [x0]
    for _ in range(20):
        xs.append(xs[-1] + V_scalar(xs[-1]))
    plt.plot(xs, "-o", ms=3, label=f"x0={x0}")
plt.axhline(2.0, ls="--", c="k", label="fixed point")
plt.xlabel("iteration"); plt.ylabel("x"); plt.legend(); plt.grid(alpha=0.3)
plt.title("x <- x + V(x) converges wherever V vanishes"); plt.show()

Note the two things the iteration needs, and *only* these two:

1. $\mathbf{V}$ vanishes at the solution — guaranteed by anti-symmetry (Notebook 1).
2. the step is a contraction — guaranteed in practice by keeping $\|\mathbf{V}\|$ small (drift normalisation and a step
   size $\eta$; Notebook 3).

Nothing requires $\mathbf{V}$ to be the gradient of anything.

## 2. Why the stop-gradient must be there

The drift $\mathbf{V}_{p,q_\theta}(\mathbf{x})$ depends on $\theta$ **twice**:

* through its argument $\mathbf{x}=f_\theta(\boldsymbol{\epsilon})$, and
* through the *distribution* $q_\theta$ that supplies the negative samples.

The second dependence is the problem. Differentiating through it would mean differentiating through
"the distribution of my own outputs" — in a minibatch implementation, through all the other samples in the batch.
Two things go wrong:

* **It is the wrong objective.** $\mathcal{L}=\|\mathbf{V}\|^2$ can be driven to zero by making the *negatives* move
  to meet the query rather than by matching $p$. The model can cheat by collapsing everything together.
* **It is intractable at scale.** The negatives are the whole batch; the field couples every sample to every other.

Let's demonstrate the cheat concretely on a 2-D problem: same field, one run with `detach`, one without.

In [ ]:
def bimodal(n, sep=2.0, sd=0.35):
    idx = torch.randint(0, 2, (n,))
    c = torch.tensor([[-sep, 0.0], [sep, 0.0]])
    return c[idx] + sd * torch.randn(n, 2)

def pdist(A, B, eps=1e-6):
    # sqrt(d^2 + eps) instead of torch.cdist: differentiable even when a point
    # is compared against itself, which happens because y_neg is x.
    return (((A[:, None, :] - B[None, :, :]) ** 2).sum(-1) + eps).sqrt()

def compute_V(x, y_pos, y_neg, tau=0.5):
    d_pos = pdist(x, y_pos)
    d_neg = pdist(x, y_neg) + torch.eye(x.shape[0], device=x.device) * 1e6
    logit = torch.cat([-d_pos / tau, -d_neg / tau], dim=1)
    A = torch.sqrt(logit.softmax(-1) * logit.softmax(-2) + 1e-20)
    P = y_pos.shape[0]
    A_pos, A_neg = A[:, :P], A[:, P:]
    return (A_pos * A_neg.sum(1, keepdim=True)) @ y_pos - (A_neg * A_pos.sum(1, keepdim=True)) @ y_neg

class Gen(nn.Module):
    def __init__(self, h=128, nl=3):
        super().__init__()
        L, d = [], 2
        for _ in range(nl):
            L += [nn.Linear(d, h), nn.SiLU()]; d = h
        L += [nn.Linear(d, 2)]
        self.net = nn.Sequential(*L)
    def forward(self, e): return self.net(e)

def train(stopgrad, steps=2000, N=256, eta=0.3, seed=0):
    torch.manual_seed(seed)
    g = Gen(); opt = torch.optim.Adam(g.parameters(), 2e-3)
    hist = []
    for _ in range(steps):
        x = g(torch.randn(N, 2))
        y_pos = bimodal(N)
        if stopgrad:
            with torch.no_grad():
                V = eta * compute_V(x, y_pos, x)
            loss = F.mse_loss(x, (x + V).detach())
        else:
            V = eta * compute_V(x, y_pos, x)        # gradients flow through the negatives too
            loss = V.pow(2).mean()                  # == the same loss value, differentiated fully
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(g.parameters(), 1.0)
        opt.step()
        hist.append(loss.item())
    with torch.no_grad():
        return g(torch.randn(4000, 2)), hist

s_sg, h_sg = train(True)
s_bp, h_bp = train(False)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ref = bimodal(4000)
for a, (s, t) in zip(ax, [(ref, "target p"), (s_sg, "with stop-gradient"), (s_bp, "backprop through V")]):
    a.scatter(s[:, 0], s[:, 1], s=3, alpha=0.35)
    a.set_title(f"{t}\nstd = {s.std(0).tolist()[0]:.2f}, {s.std(0).tolist()[1]:.2f}")
    a.set_xlim(-4, 4); a.set_ylim(-2, 2); a.set_aspect("equal")
plt.tight_layout(); plt.show()

Backpropagating through $\mathbf{V}$ collapses the model to a **single point** (standard deviation exactly zero):
it discovers that the cheapest way to make the field vanish is to put every sample in the same place, where
attraction and repulsion are both degenerate. The stop-gradient version reproduces $p$.

**The stop-gradient is not a numerical convenience. It is what makes the objective mean anything.**

A second, purely mechanical reason: with `y_neg = x`, the distance matrix has zeros on its diagonal, and
$\nabla\|\mathbf{x}-\mathbf{x}\|$ is undefined. `torch.cdist` returns `NaN` there. That is why `pdist` above uses
$\sqrt{d^2+\varepsilon}$ — needed only for this demonstration, since the real objective never differentiates the field.

## 3. The value of the loss vs. the gradient of the loss

Two facts that look contradictory until you write them out.

**Value.** $\mathcal{L}=\|\mathbf{x}-\mathrm{sg}(\mathbf{x}+\mathbf{V})\|^2 = \|\mathbf{V}\|^2$. So the number you print
during training *is* the squared drift magnitude — a direct read-out of distance from equilibrium. That's why the
loss curve in the paper's Figure 4 is meaningful in a way a diffusion loss curve is not.

**Gradient.** With the stop-gradient,
$$\frac{\partial\mathcal{L}}{\partial\mathbf{x}} = 2\big(\mathbf{x}-\mathrm{sg}(\mathbf{x}+\mathbf{V})\big) = -2\,\mathbf{V}(\mathbf{x})$$
so
$$\frac{\partial \mathcal{L}}{\partial\theta} = -2\,\mathbb{E}\!\left[\mathbf{V}(\mathbf{x})^{\!\top}\frac{\partial\mathbf{x}}{\partial\theta}\right]$$

**The drift is (minus one half of) the gradient signal handed to the output layer.** You are doing gradient descent
on nothing in particular; you are pushing outputs along a field you designed. This is the identity the paper uses in
Appendix C.2 to compare against MMD, and the identity `THEORY.md` §8 uses to convert *any* discrepancy into a drifting
field.

In [ ]:
x = torch.randn(64, 2, requires_grad=True)
y_pos = bimodal(64)
with torch.no_grad():
    V = compute_V(x, y_pos, x)
loss = F.mse_loss(x, (x + V).detach())
gx = torch.autograd.grad(loss, x)[0]

# mse_loss averages over all elements -> factor 1/(N*D)
N, D = x.shape
print("loss value          :", loss.item())
print("mean ||V||^2 / D    :", V.pow(2).sum(-1).mean().item() / D)
print("max |dL/dx + 2V/ND| :", (gx + 2 * V / (N * D)).abs().max().item())

Both identities confirmed numerically: the loss equals the mean squared drift (per element), and the gradient at the
output is exactly $-2\mathbf{V}$ up to the averaging constant.

## 4. The implicit step size

The regression target is $\mathbf{x}+\mathbf{V}$, but a single optimiser step does not *reach* it — it moves part of
the way. So the effective transport per training step is
$$\Delta\mathbf{x}_i \;=\; \underbrace{\kappa_i}_{\text{how far the optimiser got}}\;\cdot\;\mathbf{V}(\mathbf{x}_i)$$
with $\kappa_i$ depending on learning rate, optimiser state, and how well the network can represent the target.
This is why in practice you control transport speed with **two** knobs: the drift scale $\eta$ (explicit) and the
learning rate (implicit). Notebook 3 shows what happens when they are mismatched.

In [ ]:
def kappa_probe(lr, steps=400, N=256, eta=1.0, seed=0):
    torch.manual_seed(seed)
    g = Gen(); opt = torch.optim.Adam(g.parameters(), lr)
    ks = []
    for _ in range(steps):
        e = torch.randn(N, 2)
        x = g(e); y_pos = bimodal(N)
        with torch.no_grad():
            V = eta * compute_V(x, y_pos, x)
        loss = F.mse_loss(x, (x + V).detach())
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            realised = g(e) - x                       # what the net actually moved
            num = (realised * V).sum(); den = (V * V).sum().clamp_min(1e-12)
            ks.append((num / den).item())             # projection of the move onto V
    return ks

plt.figure(figsize=(6, 3.2))
for lr in [1e-4, 1e-3, 1e-2]:
    plt.plot(kappa_probe(lr), lw=1, label=f"lr={lr}")
plt.axhline(1.0, ls="--", c="k", label="fully realised target")
plt.xlabel("training step"); plt.ylabel(r"realised fraction $\kappa$")
plt.title("how much of the drifted target one optimiser step achieves")
plt.legend(); plt.grid(alpha=0.3); plt.ylim(-0.2, 1.6); plt.show()

At a small learning rate only a few percent of the drift is realised per step; the transport is slow but smooth. At a
large learning rate the network overshoots (and the noisy per-sample field makes it unstable). The paper's recipe —
gradient clip 2.0, warmup, EMA, drift normalisation — is all about keeping $\kappa$ in a benign range.

## 5. This is a self-distillation objective

The pattern "regress onto a frozen, slightly-improved copy of your own output" appears all over modern ML:

| method | frozen target |
|---|---|
| SimSiam / BYOL | the other view's embedding, stop-gradient (+ EMA teacher) |
| consistency models (iCT) | the model at an adjacent noise level |
| MeanFlow | the model's own average-velocity identity |
| **drifting** | **your own sample, displaced by the drift field** |

The differences matter. SimSiam's target is another *view*; a consistency model's target is the same model at a
different *time*; drifting's target is the same model's *sample*, moved by a field computed from the data and the
batch. That is why drifting has no time variable, no ODE, and no teacher network — the "teacher" is the data
distribution itself, consulted through a kernel.

One practical inheritance from that family: **EMA of the weights** helps a lot, because the target is noisy
(the field is a Monte-Carlo estimate). The paper uses EMA decay in $\{0.999, 0.9995, 0.9998, 0.9999\}$.

## Summary

* Drifting is a fixed-point iteration for $\mathbf{V}=\mathbf{0}$, converted into a regression by freezing the target.
* The stop-gradient blocks the path through $q_\theta$. Without it the model collapses — verified above.
* $\mathcal{L}=\|\mathbf{V}\|^2$ (a real convergence diagnostic) but $\partial\mathcal{L}/\partial\mathbf{x}=-2\mathbf{V}$
  (the field *is* the gradient signal).
* Transport speed is set by the drift scale **and** the learning rate.
* It belongs to the self-distillation family, so EMA and warmup earn their keep.

Now we have all the pieces. Next: [1. Drifting basics](<1. Drifting basics.ipynb>).